<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/llm/featherlessai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Featherless AI

[Featherless](https://featherless.ai/) is a serverless AI inference platform that makes tens of thousands of open-source AI models instantly available through a single API.

Every model on Featherless, from Qwen, DeepSeek, Kimi and GLM to fine-tunes published on Hugging Face, is served through the same OpenAI-compatible endpoint, so you can switch models by changing one string and never manage a GPU. Browse the catalog at [featherless.ai/models](https://featherless.ai/models).

## Setup

If you're opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index-llms-featherlessai

In [ ]:
!pip install llama-index

Get an API key from [featherless.ai/account/api-keys](https://featherless.ai/account/api-keys) and set it as an environment variable:

```bash
export FEATHERLESS_API_KEY=<your api key>
```

If it is not set, the cell below asks for it.

In [ ]:
import os
from getpass import getpass

if "FEATHERLESS_API_KEY" not in os.environ:
    os.environ["FEATHERLESS_API_KEY"] = getpass(
        "Enter your Featherless API key: "
    )

In [ ]:
from llama_index.llms.featherlessai import FeatherlessLLM

llm = FeatherlessLLM(model="Qwen/Qwen3.8-27B", context_window=32768)

Any model id from [featherless.ai/models](https://featherless.ai/models) works as `model`.

## Call `complete` with a prompt

In [ ]:
resp = llm.complete(
    "Explain in two sentences what an inference provider does."
)
print(resp)

An inference provider hosts machine learning models and serves them through APIs so applications can send input data and receive predictions. It handles the underlying infrastructure, scaling, and optimization needed to run models efficiently in production.


## Call `chat` with a list of messages

`chat` takes the whole conversation. Append each reply to `messages` and the model can answer follow-ups.

In [ ]:
from llama_index.core.llms import ChatMessage

messages = [
    ChatMessage(role="system", content="Answer in one or two sentences."),
    ChatMessage(role="user", content="What is a context window?"),
]
resp = llm.chat(messages)
print(resp)

messages.append(resp.message)
messages.append(
    ChatMessage(role="user", content="Why does that matter for agents?")
)
resp = llm.chat(messages)
print(resp)

assistant: A context window is the maximum amount of text—measured in tokens—that a language model can process at once. It includes both the input prompt and the model’s response, limiting how much information it can “remember” in a single interaction.
assistant: Agents often perform multi-step tasks involving tool calls, observations, and intermediate reasoning, so a limited context window can cause them to lose track of earlier instructions or results. This constrains how long and complex a task an agent can handle before it needs memory management strategies like summarization or external storage.


## Streaming

`stream_chat` streams the reply token by token.

In [ ]:
resp = llm.stream_chat(messages)
for r in resp:
    print(r.delta, end="")

Agents accumulate context over many steps—tool calls, observations, and prior reasoning—so a limited window means they can eventually "forget" earlier instructions or state, breaking long multi-step tasks. This forces design choices like summarization, external memory, or careful prompt management to keep critical information within the window.

## Agents and tool calling

Featherless supports OpenAI-style tool calling, so `FeatherlessLLM` works with LlamaIndex agents such as `FunctionAgent`. A model cannot know the current time on its own, so here the agent calls a Python function to find out.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

from llama_index.core.agent.workflow import FunctionAgent


def get_current_time(timezone: str) -> str:
    """Get the current date and time in an IANA timezone, e.g. Asia/Tokyo."""
    return datetime.now(ZoneInfo(timezone)).strftime("%A %Y-%m-%d %H:%M")


agent = FunctionAgent(tools=[get_current_time], llm=llm)
response = await agent.run("What time is it in Tokyo right now?")
print(response)

It's currently **4:01 AM on Friday, November 13, 2026** in Tokyo.


Tool calling is on by default from `llama-index-llms-featherlessai` 0.4.2. On older versions, pass `is_function_calling_model=True` to `FeatherlessLLM`.

## Choosing a model and context window

LlamaIndex uses `context_window` to decide how much chat history and retrieved context fits in a prompt, and assumes 3,900 tokens if you do not set it. Set it to what your model and Featherless plan allow. The [models endpoint](https://featherless.ai/docs/api-reference-models) lists each model's `context_length`, and whether it supports tool calling under `features.tool_use`:

```python
import httpx

models = httpx.get("https://api.featherless.ai/v1/models").json()["data"]
qwen = next(m for m in models if m["id"] == "Qwen/Qwen3.8-27B")
print(qwen["context_length"], qwen["features"])
```